In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [2]:
train_df = pd.read_csv(
    '../dataset/train_data.txt',
    sep=':::',
    engine='python',
    header=None,
    names=['id','title','genre','plot']
)

train_df.head()

,id,title,genre,plot
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous r...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...


In [3]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54214 entries, 0 to 54213
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      54214 non-null  int64
 1   title   54214 non-null  str  
 2   genre   54214 non-null  str  
 3   plot    54214 non-null  str  
dtypes: int64(1), str(3)
memory usage: 1.7 MB


In [4]:
train_df.shape

(54214, 4)

In [5]:
train_df.isnull().sum()

id       0
title    0
genre    0
plot     0
dtype: int64

In [6]:
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\VGEXC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [8]:
train_df['clean_plot'] = train_df['plot'].apply(clean_text)

train_df[['plot','clean_plot']].head()

,plot,clean_plot
0,Listening in to a conversation between his do...,listening conversation doctor parents year old...
1,A brother and sister with a past incestuous r...,brother sister past incestuous relationship cu...
2,As the bus empties the students for their fie...,bus empties students field trip museum natural...
3,To help their unemployed father make ends mee...,help unemployed father make ends meet edith tw...
4,The film's title refers not only to the un-re...,film title refers un recovered bodies ground z...


In [9]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(train_df['clean_plot'])

y = train_df['genre']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
nb = MultinomialNB()

nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

print("Naive Bayes Accuracy:",
      accuracy_score(y_test, nb_pred))

Naive Bayes Accuracy: 0.5216268560361523


In [12]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, lr_pred))

Logistic Regression Accuracy: 0.580927787512681


In [13]:
svm = LinearSVC()

svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)

print("SVM Accuracy:",
      accuracy_score(y_test, svm_pred))

SVM Accuracy: 0.5733653048049433


In [14]:
print("NB :", accuracy_score(y_test, nb_pred))
print("LR :", accuracy_score(y_test, lr_pred))
print("SVM:", accuracy_score(y_test, svm_pred))

NB : 0.5216268560361523
LR : 0.580927787512681
SVM: 0.5733653048049433


In [15]:
print(classification_report(y_test, svm_pred))

               precision    recall  f1-score   support

      action        0.42      0.32      0.36       263
       adult        0.56      0.35      0.43       112
   adventure        0.31      0.21      0.25       139
   animation        0.40      0.18      0.25       104
   biography        0.00      0.00      0.00        61
      comedy        0.52      0.57      0.55      1443
       crime        0.19      0.05      0.07       107
 documentary        0.69      0.81      0.75      2659
       drama        0.56      0.71      0.63      2697
      family        0.25      0.12      0.16       150
     fantasy        0.06      0.01      0.02        74
   game-show        0.79      0.65      0.71        40
     history        0.00      0.00      0.00        45
      horror        0.61      0.64      0.62       431
       music        0.55      0.52      0.53       144
     musical        0.00      0.00      0.00        50
     mystery        0.18      0.05      0.08        56
        n

In [16]:
import joblib

joblib.dump(svm,
            '../models/movie_genre_model.pkl')

joblib.dump(tfidf,
            '../models/tfidf.pkl')

['../models/tfidf.pkl']